***Шалабанов Александр - Notebook для создания модели прогнозирования***

Загружаем библиотеки

In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.__version__, pd.__version__

('2.5.5', '3.1.1')

Загрузка данных из папки data

In [66]:
data = pd.read_csv("data/train.csv")
print(data.shape)
data.head()

(1656, 37)


,sku,unit,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,...,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11
0,0,ШТ,128.0,125.0,179.0,199.0,204.0,246.0,172.0,235.0,...,379.0,250.0,317.0,172.0,193.0,173.0,121.0,318.0,328.0,208.0
1,1,ШТ,65.0,64.0,54.0,91.0,74.0,91.0,138.0,115.0,...,68.0,65.0,72.0,82.0,85.0,69.0,75.0,100.0,72.0,89.0
2,2,ШТ,157.0,205.0,210.0,205.0,265.0,248.0,292.0,268.0,...,334.0,380.0,409.0,309.0,388.0,438.0,404.0,293.0,274.0,381.0
3,3,ШТ,2450.0,2280.0,2144.0,2283.0,2177.0,2888.0,3066.0,2288.0,...,2806.0,2085.0,1984.0,2086.0,2871.0,2204.0,1821.0,2187.0,1672.0,1665.0
4,4,ШТ,119.0,154.0,154.0,270.0,513.0,718.0,720.0,292.0,...,222.0,258.0,289.0,662.0,759.0,616.0,365.0,321.0,392.0,409.0


In [67]:
testData = pd.read_csv("data/test.csv")
print(testData.shape)
testData.head()

(4968, 4)


,id,sku,unit,month
0,0_2021-12,0,ШТ,2021-12
1,0_2022-01,0,ШТ,2022-01
2,0_2022-02,0,ШТ,2022-02
3,1_2021-12,1,ШТ,2021-12
4,1_2022-01,1,ШТ,2022-01


Разобьём наш Data в удобный вид разделив их по времени

In [83]:
class SplitDateOnColumn:
    def fit(self, dataFrame):
        pass
    def transform(self, dataFrame):
        resDataDictColls = {"sku": [], "unit": [], "month": [], "resultValue": []}
        
        for serialId in range(0, len(dataFrame)):
            serial = dataFrame.values[serialId]
            for i in range(2, len(dataFrame.columns)):
                resDataDictColls["sku"] += [serial[0]]
                resDataDictColls["unit"] += [serial[1]]
                resDataDictColls["month"] += [dataFrame.columns[i]]
                resDataDictColls["resultValue"] += [serial[i]]
        result = pd.DataFrame(resDataDictColls)
        return result
    def fit_transform(self, dataFrame):
        return self.transform(dataFrame)

splitter = SplitDateOnColumn()
splittedData = splitter.transform(data)
splittedData.head()

,sku,unit,month,resultValue
0,0,ШТ,2019-01,128.0
1,0,ШТ,2019-02,125.0
2,0,ШТ,2019-03,179.0
3,0,ШТ,2019-04,199.0
4,0,ШТ,2019-05,204.0


Далее для ещё более удобно записи и прогнозирования разделим месяцы на год и месяц. Написать его следует так, чтобы и тестовые данные можно было обработать

In [84]:
class SplitDateOnYearsMonth:
    def fit(self, dataFrame):
        pass
    def transform(self, dataFrame):
        dateData = dataFrame.loc[..., "month"]
        resDict = {"years": [], "month" : []}
        for dates in dateData:
            splittedText = dates.split("-")
            resDict["years"] += [int(splittedText[0])]
            resDict["month"] += [int(splittedText[1])]
        df = pd.DataFrame(resDict)
        df = dataFrame.drop("month", axis=1).join(df)
        return df
    def fit_transform(self, dataFrame):
        return self.transform(dataFrame)
dateSplitter = SplitDateOnYearsMonth()
transformedTrainData = dateSplitter.transform(splittedData)
transformedTrainData

,sku,unit,resultValue,years,month
0,0,ШТ,128.0,2019,1
1,0,ШТ,125.0,2019,2
2,0,ШТ,179.0,2019,3
3,0,ШТ,199.0,2019,4
4,0,ШТ,204.0,2019,5
...,...,...,...,...,...
57955,1655,ШТ,121.0,2021,7
57956,1655,ШТ,146.0,2021,8
57957,1655,ШТ,123.0,2021,9
57958,1655,ШТ,124.0,2021,10


Данные мы преобразовали теперь можно начать с ними работать, начнём с разведочного анализа TO BE CONTINUE